EXERCICE 1 — Data Loading and Preparation

In [1]:
# Cellule 1 — Installation
!pip install -q faiss-cpu==1.7.4
!pip install -qU chromadb
!pip install -q numpy
!apt install libomp-dev -y --quiet
!python -m pip install --upgrade faiss-cpu --quiet

import os
os.makedirs("cache", exist_ok=True)
print("Setup terminé !")

ERROR: Could not find a version that satisfies the requirement faiss-cpu==1.7.4 (from versions: 1.8.0, 1.8.0.post1, 1.9.0, 1.9.0.post1, 1.10.0, 1.11.0, 1.11.0.post1, 1.12.0, 1.13.0, 1.13.1, 1.13.2, 1.14.2, 1.14.3)
ERROR: No matching distribution found for faiss-cpu==1.7.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 145.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━

In [2]:
# Cellule 2 — Imports
import numpy as np
import pandas as pd
import faiss
import json
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("Imports OK !")

Imports OK !


In [7]:
import pandas as pd

# Cellule 3 — Charger le dataset
# Télécharge le fichier labelled_newscatcher_dataset.csv
# et upload-le dans Colab, puis :

path = "/content/sample_data/labelled_newscatcher_dataset.csv"
pdf  = pd.read_csv(path, sep=";")

# Ajouter une colonne ID unique
pdf["id"] = pdf.index.astype(str)

# Inspecter les données
display(pdf.head())
print(f"\nShape : {pdf.shape}")
print(f"\nTypes : {pdf.dtypes}")
print(f"Colonnes : {pdf.columns.tolist()}")
print(f"\nValeurs manquantes :\n{pdf.isnull().sum()}")

,topic,link,domain,published_date,title,lang,id
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en,0
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en,1
2,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en,2
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en,3
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en,4



Shape : (108774, 7)

Types : topic             object
link              object
domain            object
published_date    object
title             object
lang              object
id                object
dtype: object
Colonnes : ['topic', 'link', 'domain', 'published_date', 'title', 'lang', 'id']

Valeurs manquantes :
topic             0
link              0
domain            0
published_date    0
title             0
lang              0
id                0
dtype: int64


In [8]:
# Cellule 4 — Créer un subset de 1000 lignes
pdf_subset = pdf.head(1000).copy()

print(f"Subset créé : {pdf_subset.shape}")
display(pdf_subset.head())

Subset créé : (1000, 7)


,topic,link,domain,published_date,title,lang,id
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en,0
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en,1
2,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en,2
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en,3
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en,4


EXERCICE 2 — Vectorisation avec Sentence Transformers

In [9]:
# Cellule 5 — Préparer les données
from sentence_transformers import SentenceTransformer, InputExample

def example_create_fn(doc1: str) -> InputExample:
    """
    Convertit un titre en InputExample pour SentenceTransformer.
    doc1 : le titre de l'article
    """
    return InputExample(texts=[doc1])

# Appliquer sur le subset
faiss_train_examples = pdf_subset.apply(
    lambda x: example_create_fn(x["title"]), axis=1
).tolist()

print(f"Nombre d'exemples créés : {len(faiss_train_examples)}")
print(f"Premier exemple : {faiss_train_examples[0].texts}")

Nombre d'exemples créés : 1000
Premier exemple : ["A closer look at water-splitting's solar fuel potential"]


In [10]:
# Cellule 6 — Initialiser le modèle d'embedding
model = SentenceTransformer('all-MiniLM-L6-v2')

# Extraire les titres
titles_list = pdf_subset["title"].tolist()
print(f"Nombre de titres : {len(titles_list)}")
print(f"Exemple titre : {titles_list[0]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Nombre de titres : 1000
Exemple titre : A closer look at water-splitting's solar fuel potential


In [11]:
# Cellule 7 — Générer les embeddings
print("Génération des embeddings en cours...")
faiss_title_embedding = model.encode(titles_list, show_progress_bar=True)

# Vérifier les dimensions
print(f"\nNombre d'embeddings : {len(faiss_title_embedding)}")
print(f"Dimension par embedding : {len(faiss_title_embedding[0])}")
# → 1000 embeddings de 384 dimensions chacun

Génération des embeddings en cours...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


Nombre d'embeddings : 1000
Dimension par embedding : 384


EXERCICE 3 — FAISS Indexing and Search

In [14]:
# Cellule 1 — Installation
!pip install -q faiss-cpu==1.7.4
!pip install -qU chromadb
!pip install -q numpy
!apt install libomp-dev -y --quiet
!python -m pip install --upgrade faiss-cpu --quiet

import os
os.makedirs("cache", exist_ok=True)
print("Setup terminé !")

# Cellule 8 — Préparer et normaliser les vecteurs
import numpy as np
import faiss

pdf_to_index = pdf_subset.copy()
id_index     = np.array(pdf_to_index.index.tolist())

# Copier les embeddings et les convertir en float32
content_encoded_normalized = np.array(faiss_title_embedding).astype('float32')

# Normaliser pour la similarité cosinus
faiss.normalize_L2(content_encoded_normalized)

print(f"Vecteurs normalisés : {content_encoded_normalized.shape}")

ERROR: Could not find a version that satisfies the requirement faiss-cpu==1.7.4 (from versions: 1.8.0, 1.8.0.post1, 1.9.0, 1.9.0.post1, 1.10.0, 1.11.0, 1.11.0.post1, 1.12.0, 1.13.0, 1.13.1, 1.13.2, 1.14.2, 1.14.3)
ERROR: No matching distribution found for faiss-cpu==1.7.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━

In [15]:
# Cellule 9 — Créer l'index FAISS
dimension     = len(faiss_title_embedding[0])  # 384
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(dimension))
index_content.add_with_ids(content_encoded_normalized, id_index)

print(f"Dimension de l'index : {dimension}")
print(f"Nombre de vecteurs indexés : {index_content.ntotal}")

Dimension de l'index : 384
Nombre de vecteurs indexés : 1000


In [16]:
# Cellule 10 — Fonction de recherche
def search_content(query, pdf_to_index, k=3):
    """
    Recherche les articles les plus similaires à une requête.

    query         : question ou phrase de recherche
    pdf_to_index  : DataFrame contenant les articles originaux
    k             : nombre de résultats à retourner
    """
    # Encoder la requête
    query_vector = model.encode([query]).astype('float32')

    # Normaliser le vecteur requête
    faiss.normalize_L2(query_vector)

    # Chercher les k plus proches voisins
    top_k = index_content.search(query_vector, k)
    ids           = top_k[1][0].tolist()
    similarities  = top_k[0][0].tolist()

    # Récupérer les articles correspondants
    results = pdf_to_index[pdf_to_index.index.isin(ids)].copy()
    results["similarities"] = similarities[:len(results)]

    return results

# Test
display(search_content("animal", pdf_to_index, k=5))
display(search_content("technology", pdf_to_index, k=5))

,topic,link,domain,published_date,title,lang,id,similarities
99,TECHNOLOGY,https://www.gematsu.com/2020/08/ghostwire-toky...,gematsu.com,2020-08-07 16:43:13,Ghostwire: Tokyo confirms dog petting,en,99,0.391902
176,TECHNOLOGY,https://www.pushsquare.com/news/2020/08/random...,pushsquare.com,2020-08-03 16:30:00,Random: You Can Pick Up and Pet Cats in Assass...,en,176,0.376784
762,SCIENCE,https://af.reuters.com/article/worldNews/idAFK...,af.reuters.com,2020-08-13 16:51:00,'Secret' life of sharks: Study reveals their s...,en,762,0.344059
928,SCIENCE,https://www.thecut.com/2020/08/scientists-say-...,thecut.com,2020-08-04 12:52:00,Just Let This Lizard Be a Dinosaur,en,928,0.317387
975,HEALTH,https://www.news-medical.net/news/20200813/Res...,news-medical.net,2020-08-13 05:18:00,Researchers explore social behavior of animals...,en,975,0.295497


,topic,link,domain,published_date,title,lang,id,similarities
198,SCIENCE,https://www.unilad.co.uk/technology/scientists...,unilad.co.uk,2020-08-17 14:29:00,Scientists Discover New Material That Could ‘M...,en,198,0.452841
224,SCIENCE,https://swordstoday.ie/this-is-when-earth-will...,swordstoday.ie,2020-08-13 20:23:21,This is when Earth will not be in a position t...,en,224,0.298307
418,TECHNOLOGY,https://www.kotaku.com.au/2020/08/the-fascinat...,kotaku.com.au,2020-08-13 23:32:00,"The Fascinating Web Of Entropia Universe, The ...",en,418,0.293503
453,TECHNOLOGY,https://www.helpnetsecurity.com/2020/08/05/way...,helpnetsecurity.com,2020-08-05 03:30:00,Ways AI could be used to facilitate crime over...,en,453,0.291862
716,TECHNOLOGY,https://www.lightreading.com/services/eurobite...,lightreading.com,2020-08-07 11:35:07,"Eurobites: UK is (almost) totally wired, finds...",en,716,0.291742


EXERCICE 4 — ChromaDB

In [22]:
!pip install -qU chromadb
!pip install -q numpy
!apt install libomp-dev -y --quiet
!python -m pip install --upgrade faiss-cpu --quiet

import os
os.makedirs("cache", exist_ok=True)
print("Setup terminé !")


# Cellule 11 — ChromaDB
import chromadb
from chromadb.config import Settings

chroma_client   = chromadb.Client()
collection_name = "my_news"

# Supprimer si existe déjà
existing = [c.name for c in chroma_client.list_collections()]
if collection_name in existing:
    chroma_client.delete_collection(name=collection_name)

print(f"Création de la collection : '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)
print("Collection créée !")

Reading package lists...
Building dependency tree...
Reading state information...
libomp-dev is already the newest version (1:14.0-55~exp2).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
Setup terminé !
Création de la collection : 'my_news'
Collection créée !


In [23]:
# Cellule 12 — Ajouter les 100 premiers articles
collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": str(topic)} for topic in pdf_subset["topic"][:100].tolist()],
    ids=[str(i) for i in range(100)]  # IDs uniques sous forme de strings
)

print(f"100 articles ajoutés à ChromaDB !")
print(f"Total dans la collection : {collection.count()}")

100 articles ajoutés à ChromaDB !
Total dans la collection : 100


In [27]:
import json

# Cellule 13 — Requête ChromaDB
results = collection.query(
    query_texts=["space"],
    n_results=10
)

print(json.dumps(results, indent=4))

{
    "ids": [
        [
            "72",
            "7",
            "30",
            "26",
            "23",
            "76",
            "69",
            "40",
            "47",
            "75"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "Beck teams up with NASA and AI for 'Hyperspace' visual album experience",
            "Orbital space tourism set for rebirth in 2021",
            "NASA drops \"insensitive\" nicknames for cosmic objects",
            "\u2018It came alive:\u2019 NASA astronauts describe experiencing splashdown in SpaceX Dragon",
            "Hubble Uses Moon As \u201cMirror\u201d to Study Earth\u2019s Atmosphere \u2013 Proxy in Search of Potentially Habitable Planets Around Other Stars",
            "Australia's small yet crucial part in the mission to find life on Mars",
            "NASA Astronauts in SpaceX Capsule Splashdown in Gulf Of Mexico",
            "SpaceX's Starship spacecraft saw 150 meters high",
          

EXERCICE 5 — Question Answering avec HuggingFace

In [28]:
# Cellule 14 — Charger GPT-2
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id  = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model  = AutoModelForCausalLM.from_pretrained(model_id)

# Créer le pipeline
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    device_map="auto",
    pad_token_id=tokenizer.eos_token_id
)

print("Modèle prêt !")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Modèle prêt !


In [29]:
# Cellule 15 — RAG complet : ChromaDB + GPT-2
question = "What's the latest news on space development?"

# Récupérer le contexte depuis ChromaDB
context_results = collection.query(
    query_texts=[question],
    n_results=5
)

context = " ".join([
    f"#{str(i)}" for i in context_results["documents"][0]
])

# Construire le prompt
prompt_template = f"Relevant context: {context}\n\nThe user's question: {question}\n\nAnswer:"

# Générer la réponse
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

[transformers] Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Relevant context: #Orbital space tourism set for rebirth in 2021 #Beck teams up with NASA and AI for 'Hyperspace' visual album experience #SpaceX, NASA Demo-2 Rocket Launch Set for Saturday: How to Watch #Xbox Series X will release no later than November 13, based on this listing #‘It came alive:’ NASA astronauts describe experiencing splashdown in SpaceX Dragon

The user's question: What's the latest news on space development?

Answer:

"We're in the early development stages. But we've been working with NASA for a while on a number of projects that go beyond just just getting the hardware and software. We also like to work with partners to make sure we're making the right decisions and making the right investments. We're working with some of the best teams in the world to make sure that we're doing the right thing for the customers."

The user's question: What's the latest news on space development?

Answer:

"We're in the early development stage. But we've been working with NASA for 